In [ ]:
!sudo apt-get install -y fonts-nanum
# 폰트 설치 후 Colab 메뉴에서 런타임/세션 다시 시작을 선택합니다.

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
!pip install -U openai

In [ ]:
from google.colab import userdata
import os
import openai

# Colab에 저장한 Secret에서 API 키 가져와 환경 변수에 등록
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = openai.OpenAI()

In [ ]:
def make_evaluation_prompt(prompt, answer):
    return f"""
        프롬프트: {prompt}
        응답: {answer}

        위 프롬프트에 대한 응답을 정답/오답으로 답변해주세요.
        정답/오답만 답변합니다.
    """

In [ ]:
prompts_and_answers = [
    ("프랑스의 수도는 어디인가요?", "파리"),
    ("전세계에서 가장 높은 산은 어디인가요?", "에베레스트"),
    ("1 더하기 3은 얼마인가요?", "5")
]

for prompt, answer in prompts_and_answers:
    eval_prompt = make_evaluation_prompt(prompt, answer)

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": eval_prompt}
        ]
    )

    content = response.choices[0].message.content
    print(f"Prompt: {prompt}, Answer: {answer}, Evaluation: {content}")

    if content == '오답':
        score = {"correct": False}
    else:
        score = {"correct": True}
    # You can do something with the score here, e.g., store it in a list
    print(f"Score: {score}")

Prompt: 프랑스의 수도는 어디인가요?, Answer: 파리, Evaluation: 정답
Score: {'correct': True}
Prompt: 전세계에서 가장 높은 산은 어디인가요?, Answer: 에베레스트, Evaluation: 정답
Score: {'correct': True}
Prompt: 1 더하기 3은 얼마인가요?, Answer: 5, Evaluation: 오답
Score: {'correct': False}


In [ ]:
results = []
for prompt, answer in prompts_and_answers:
    # 프롬프트에 대한 답변이 맞았는지 평가를 요청
    eval_prompt = make_evaluation_prompt(prompt, answer)  # 프롬프트와 응답으로 평가 프롬프트 생성
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role": "user", "content": eval_prompt}
        ]
    )

    content = response.choices[0].message.content
    print(content)

    if content == '오답':
        score = {"correct": False}
    else:
        score = {"correct": True}
    score["prompt"] = prompt  # 원본 프롬프트나 식별자 저장
    score["answer"] = answer
    score["model"] = "gpt-4o-mini"
    results.append(score)

정답
정답
오답


In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df  # 데이터프레임 출력

,correct,prompt,answer,model
0,True,프랑스의 수도는 어디인가요?,파리,gpt-4o-mini
1,True,전세계에서 가장 높은 산은 어디인가요?,에베레스트,gpt-4o-mini
2,False,1 더하기 3은 얼마인가요?,5,gpt-4o-mini
